<a href="https://colab.research.google.com/github/NayanNair18/mgmt467-analytics-portfolio/blob/main/Lab2_Advanced_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab: Vertex AI–Assisted BigQuery Analytics — Example Prompts
**Goal:** Practice moving from simple SQL to complex analytics in BigQuery using *only* carefully engineered prompts with Vertex AI (Gemini).  
**Important:** This notebook contains **prompts only** (no starter code). Paste the prompts into **Vertex AI Studio**, **Vertex AI in Colab Enterprise**, or your chosen chat interface, and then run the generated SQL directly in **BigQuery**. If you decide to automate later, you can ask Vertex AI to convert the winning SQL into a Colab pipeline.

## How to use this prompts-only notebook
1. Open **Vertex AI Studio** (or Gemini in Colab Enterprise chat panel).  
2. Copy a prompt from this notebook and paste it into the model. Do **not** paste any code from here; let the model generate it.  
3. Run the generated SQL in **BigQuery** (Console → BigQuery Studio).  
4. Iterate: refine the prompt when results aren’t what you expect.  
5. Document: capture your final SQL, plus a one-sentence takeaway, in your notes/README.

## Dataset assumptions
Use one of these sources (adjust table paths accordingly):
- **Global Superstore (Kaggle)** loaded into BigQuery (e.g., `[YOUR_PROJECT].superstore_data.sales`)  
- **TheLook eCommerce** public dataset: `bigquery-public-data.thelook_ecommerce`  
If you are using *Global Superstore*, make sure column names match your schema (e.g., `Order_Date`, `Region`, `Category`, `Sub_Category`, `Sales`, `Profit`, `Discount`, `State`, `Customer_ID`, `Ship_Mode`).

---
## Prompting guardrails (quick checklist)
- **Be explicit**: table path, column names, filters, output columns, sort order, and limits.  
- **Ask for runnable SQL**: “Return a BigQuery SQL block only.”  
- **Control cost**: ask for `LIMIT` during exploration and remove it for the final run.  
- **Validate**: request a brief explanation of why each clause is present and how you can sanity-check results.
---

## Install Dependencies

In [1]:
# Install the Google Cloud BigQuery client library
!pip install google-cloud-bigquery==3.17.0 pandas==2.1.4

# Authenticate your Colab environment
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

Authenticated


## Copy Schema to a dataframe

In [2]:
from google.cloud import bigquery
import pandas as pd

# Use TheLook eCommerce public dataset
project_id = 'big-data-analysis-472319'
dataset_id = 'superstore_data'
table_id = 'sales' # Using 'order_items' as an example table, you might need to adjust based on your analysis

# Construct a BigQuery client object.
client = bigquery.Client(project=project_id)

# Get the table object
table_ref = client.dataset(dataset_id).table(table_id)
table = client.get_table(table_ref)

# Extract schema information
schema_list = []
for field in table.schema:
    schema_list.append({
        'name': field.name,
        'field_type': field.field_type,
        'mode': field.mode,
        'description': field.description
    })

# Convert to Pandas DataFrame
schema_df = pd.DataFrame(schema_list)

# Display the schema DataFrame (optional, for verification)
print("Schema DataFrame created:")
# To see the output, run the code.

Schema DataFrame created:


## CLean Column Names

In [3]:
# --- 1. Clean the Column Names ---
# Create a 'clean_name' column with standard naming conventions:
# lowercase, with spaces and hyphens replaced by underscores.
schema_df['clean_name'] = schema_df['name'].str.lower().str.replace(' ', '_').str.replace('-', '_')


# --- 2. Generate the Aliases for the SELECT Clause ---
column_expressions = []
for index, row in schema_df.iterrows():
    original_name = row['name']
    clean_name = row['clean_name']

    # If the original name contains a space or special character, it needs to be
    # enclosed in backticks (`) in the SQL statement.
    if ' ' in original_name or '-' in original_name:
        expression = f'`{original_name}` AS {clean_name}'
    else:
        # If the name is already clean, we still alias it for consistency.
        expression = f'{original_name} AS {clean_name}'
    column_expressions.append(expression)

# Join all the individual column expressions into a single, formatted string.
select_clause = ",\n  ".join(column_expressions)


# --- 3. Construct the Final CREATE VIEW Statement ---
new_view_id = 'superstore_clean' # You can change this if you like

# Get the user's project ID to create the view in their project
# Assuming the user has authenticated and the project ID is available
# If not, you might need to add a step to get the project ID from the user or environment
user_project_id = client.project # Use the project ID from the authenticated client

create_view_sql = f"""
CREATE OR REPLACE VIEW `{user_project_id}.{dataset_id}.{new_view_id}` AS
SELECT
  {select_clause}
FROM
  `{project_id}.{dataset_id}.{table_id}`;
"""

# --- 4. Print the Final SQL ---
print("--- Copy the SQL below and run it in your BigQuery Console ---")
print(create_view_sql)

--- Copy the SQL below and run it in your BigQuery Console ---

CREATE OR REPLACE VIEW `big-data-analysis-472319.superstore_data.superstore_clean` AS
SELECT
  Category AS category,
  City AS city,
  Country AS country,
  Customer_ID AS customer_id,
  Customer_Name AS customer_name,
  Discount AS discount,
  Market AS market,
  _________ AS _________,
  Order_Date AS order_date,
  Order_ID AS order_id,
  Order_Priority AS order_priority,
  Product_ID AS product_id,
  Product_Name AS product_name,
  Profit AS profit,
  Quantity AS quantity,
  Region AS region,
  Row_ID AS row_id,
  Sales AS sales,
  Segment AS segment,
  Ship_Date AS ship_date,
  Ship_Mode AS ship_mode,
  Shipping_Cost AS shipping_cost,
  State AS state,
  Sub_Category AS sub_category,
  Year AS year,
  Market2 AS market2,
  weeknum AS weeknum
FROM
  `big-data-analysis-472319.superstore_data.sales`;



## Generate View with standard column naming convention

Before running the code below, **ensure the `dataset_id` in the previous cell (`hjxWwOPYgyu3`) is set to a dataset that exists within your GCP project.** You may need to create a new dataset in your project via the BigQuery console if you don't have one.

## Generate View with standard column naming convention

In [4]:
# Execute the CREATE VIEW SQL query
# Note: You should run the CREATE VIEW SQL generated in the previous cell
# in your BigQuery console, not directly in this notebook cell, due to permissions.
# This cell will now attempt to query the view you created.

try:
    # Construct a reference to the new view in the user's project
    user_project_id = client.project
    # Use a SELECT query to fetch data from the view
    query_string = f"""
    SELECT
      *
    FROM
      `{user_project_id}.{dataset_id}.{new_view_id}`
    LIMIT 10;
    """

    print(f"\n--- First 10 rows from the new view '{new_view_id}' in project '{user_project_id}' ---")
    query_job = client.query(query_string)
    rows = query_job.result() # Waits for job to complete.

    # Print header
    print(" | ".join([field.name for field in rows.schema]))
    print("-" * 80) # Separator

    # Print rows
    for row in rows:
        print(" | ".join([str(item) for item in row.values()]))

except Exception as e:
    print(f"An error occurred while fetching rows from the view: {e}")


--- First 10 rows from the new view 'superstore_clean' in project 'big-data-analysis-472319' ---
category | city | country | customer_id | customer_name | discount | market | _________ | order_date | order_id | order_priority | product_id | product_name | profit | quantity | region | row_id | sales | segment | ship_date | ship_mode | shipping_cost | state | sub_category | year | market2 | weeknum
--------------------------------------------------------------------------------
Office Supplies | Ajman | United Arab Emirates | PO-88653 | Patrick O'Donnell | 0.7 | EMEA | 1 | 2011-10-03 00:00:00.000 | AE-2011-9160 | Medium | OFF-FEL-10001405 | Fellowes File Cart, Industrial | -157.086 | 2 | EMEA | 48313 | 83 | Consumer | 2011-10-07 00:00:00.000 | Standard Class | 5.69 | 'Ajman | Storage | 2011 | EMEA | 41
Technology | Ajman | United Arab Emirates | PO-88653 | Patrick O'Donnell | 0.7 | EMEA | 1 | 2011-10-03 00:00:00.000 | AE-2011-9160 | Medium | TEC-EPS-10004171 | Epson Calculator, Red | -8

In [5]:
# This assumes your 'client' object from the previous cell is still active
# and correctly authenticated.

print("✅ Step 1: Defining the query string...")

query_string = """
SELECT
  order_id,
  customer_name,
  product_name,
  sales,
  profit
FROM
  `big-data-analysis-472319.superstore_data.superstore_clean`
LIMIT 10;
"""

print("✅ Step 2: Sending the query to BigQuery. This may take a moment...")

# Use a try-except block to catch potential errors
try:
    query_job = client.query(query_string)

    print("✅ Step 3: Waiting for query to complete and fetching results...")
    results_df = query_job.to_dataframe()

    print(f"✅ Step 4: Query finished. Found {len(results_df)} rows.")

    if results_df.empty:
        print("\n⚠️ The query ran successfully but returned an empty result. Please double-check that your 'superstore_clean' view exists and the original table has data.")
    else:
        print("\n--- Displaying Results ---")
        display(results_df)

except Exception as e:
    print(f"\n❌ An error occurred: {e}")

✅ Step 1: Defining the query string...
✅ Step 2: Sending the query to BigQuery. This may take a moment...
✅ Step 3: Waiting for query to complete and fetching results...
✅ Step 4: Query finished. Found 10 rows.

--- Displaying Results ---


,order_id,customer_name,product_name,sales,profit
0,AE-2011-9160,Patrick O'Donnell,"Fellowes File Cart, Industrial",83,-157.086
1,AE-2011-9160,Patrick O'Donnell,"Epson Calculator, Red",78,-88.992
2,JO-2011-1740,Scot Wooten,"Binney & Smith Canvas, Fluorescent",53,9.990
3,JO-2011-1740,Scot Wooten,"Rogers Folders, Blue",31,0.600
4,JO-2011-1740,Scot Wooten,"Sauder Library with Doors, Metal",387,127.710
5,SA-2011-1630,Dennis Pardue,"Elite Box Cutter, Easy Grip",37,5.460
6,SA-2011-1630,Dennis Pardue,"Eaton Parchment Paper, Multicolor",30,6.540
7,SA-2011-4390,Lena Cacioppo,"Advantus Paper Clips, Metal",13,3.210
8,SA-2011-4390,Lena Cacioppo,"Stiletto Box Cutter, Serrated",196,90.000
9,SA-2011-1630,Dennis Pardue,"Hon Bag Chairs, Black",171,34.080


## Part A — SQL Warm‑Up (SELECT, WHERE, ORDER BY, LIMIT, DISTINCT)
**Aim:** Build confidence with precise, unambiguous prompts that yield clean, runnable SQL.

### A1. Unique values (DISTINCT)
**Prompt (paste in Vertex AI):**
```
Act as a senior BigQuery analyst. Produce a **single runnable BigQuery SQL** (no commentary) for:
- Task: List all unique `Sub_Category` values sold in the 'West' region.
- Table: `mgmt-467-47888.lab1_foundation.superstore`
- Filter: `Region = 'West'`
- Output: a single column named `Sub_Category`
- Sort: alphabetically A→Z
- Add: `LIMIT 100` to control cost during exploration.
```
**Reflection:** Did the result match your expectations? If not, what ambiguity in your prompt might have caused the mismatch?

In [6]:
# Paste the BigQuery SQL generated by Vertex AI here.
# Example SQL based on the prompt:
query_string = """
SELECT
    DISTINCT sub_category
FROM
    `big-data-analysis-472319.superstore_data.superstore_clean`
WHERE
    region = 'West'
ORDER BY
    sub_category ASC
LIMIT 100
"""

# Assuming 'client' object is already initialized from previous cells
# from google.cloud import bigquery
# client = bigquery.Client(project='your-gcp-project-id') # Ensure client is initialized if not already

try:
    print("Executing BigQuery query...")
    query_job = client.query(query_string)

    # Wait for the job to complete and get the results
    results_df = query_job.to_dataframe()

    print("Query results:")
    if results_df.empty:
        print("No results found.")
    else:
        display(results_df)

except Exception as e:
    print(f"An error occurred: {e}")

Executing BigQuery query...
Query results:


,sub_category
0,Accessories
1,Appliances
2,Art
3,Binders
4,Bookcases
5,Chairs
6,Copiers
7,Envelopes
8,Fasteners
9,Furnishings


### A2. Top‑N by metric (ORDER BY … DESC)
**Prompt:**
```
BigQuery SQL only.
Task: Return the top 10 customers by total profit.
Table: `mgmt-467-47888.lab_foundation.superstore`
Columns used: `Customer_ID`, `Profit`
Output columns: `Customer_ID`, `total_profit`
Logic: SUM Profit per customer, order by `total_profit` DESC
Add `LIMIT 10`.
```
**Tip:** If your schema uses different identifiers (e.g., `Customer Name`), restate column names explicitly.

In [7]:
# Paste the BigQuery SQL generated by Vertex AI here.
# Example SQL based on the prompt:
query_string = """
SELECT
    customer_id,
    SUM(profit) AS total_profit
FROM
    `big-data-analysis-472319.superstore_data.superstore_clean`
GROUP BY
    customer_id
ORDER BY
    total_profit DESC
LIMIT 10
"""

# Assuming 'client' object is already initialized from previous cells
# from google.cloud import bigquery
# client = bigquery.Client(project='your-gcp-project-id') # Ensure client is initialized if not already

try:
    print("Executing BigQuery query...")
    query_job = client.query(query_string)

    # Wait for the job to complete and get the results
    results_df = query_job.to_dataframe()

    print("Query results:")
    if results_df.empty:
        print("No results found.")
    else:
        display(results_df)

except Exception as e:
    print(f"An error occurred: {e}")

Executing BigQuery query...
Query results:


,customer_id,total_profit
0,TC-209804,8981.3239
1,RB-193604,6976.0959
2,SC-200954,5757.4119
3,HL-150404,5622.4292
4,AB-101054,5444.8055
5,SP-209202,4974.5130
6,TA-213854,4703.7883
7,CA-127751,4045.8780
8,PJ-188352,3986.0040
9,CM-123854,3899.8904


### A3. Basic filtering (WHERE) + sanity checks
**Prompt:**
```
BigQuery SQL only.
Task: Count orders shipped with each `Ship_Mode`, but only for orders in the 'Technology' category.
Table: `[YOUR_PROJECT].superstore_data.sales`
Output: `Ship_Mode`, `order_count`
Logic: COUNT(*) grouped by `Ship_Mode`
Sort by `order_count` DESC
```
**Validation ask:** “Also list two quick sanity checks to verify the numbers.”

In [8]:
# Paste the BigQuery SQL generated by Vertex AI here.
# Example SQL based on the prompt:
query_string = """
SELECT
    ship_mode,
    COUNT(*) AS order_count
FROM
    `big-data-analysis-472319.superstore_data.superstore_clean`
WHERE
    category = 'Technology'
GROUP BY
    ship_mode
ORDER BY
    order_count DESC
"""

# Assuming 'client' object is already initialized from previous cells
# from google.cloud import bigquery
# client = bigquery.Client(project='your-gcp-project-id') # Ensure client is initialized if not already

try:
    print("Executing BigQuery query...")
    query_job = client.query(query_string)

    # Wait for the job to complete and get the results
    results_df = query_job.to_dataframe()

    print("Query results:")
    if results_df.empty:
        print("No results found.")
    else:
        display(results_df)

except Exception as e:
    print(f"An error occurred: {e}")

Executing BigQuery query...
Query results:


,ship_mode,order_count
0,Standard Class,6117
1,Second Class,2030
2,First Class,1476
3,Same Day,518


## Part B — Grouped Analytics (GROUP BY, HAVING)
**Aim:** Turn raw facts into grouped metrics and filtered aggregations.

### B1. KPI aggregation with WHERE + GROUP BY
**Prompt:**
```
BigQuery SQL only.
Task: Compute monthly revenue for the last 12 full months.
Table: `[YOUR_PROJECT].superstore_data.sales`
Assume: `Order_Date` is a DATE or TIMESTAMP column named exactly `Order_Date`.
Output: `year_month` (YYYY-MM format), `monthly_revenue`
Logic: Truncate date to month, SUM `Sales`, filter to last 12 full months.
Sort by `year_month` ascending.
Include a `LIMIT` safeguard for exploration.
```

In [9]:
# Paste the BigQuery SQL generated by Vertex AI here.
# Example SQL based on the prompt:
query_string = """
SELECT
    FORMAT_DATE('%Y-%m', DATE_TRUNC(PARSE_DATE('%Y-%m-%d', SUBSTR(order_date, 1, 10)), MONTH)) AS year_month,
    SUM(sales) AS monthly_revenue
FROM
    `big-data-analysis-472319.superstore_data.superstore_clean`
WHERE
    PARSE_DATE('%Y-%m-%d', SUBSTR(order_date, 1, 10)) BETWEEN PARSE_DATE('%Y-%m-%d', '2011-01-01') AND PARSE_DATE('%Y-%m-%d', '2012-12-31')
GROUP BY
    year_month
ORDER BY
    year_month ASC
-- LIMIT safeguard for exploration
LIMIT 100
"""

# Assuming 'client' object is already initialized from previous cells
# from google.cloud import bigquery
# client = bigquery.Client(project='your-gcp-project-id') # Ensure client is initialized if not already

try:
    print("Executing BigQuery query...")
    query_job = client.query(query_string)

    # Wait for the job to complete and get the results
    results_df = query_job.to_dataframe()

    print("Query results:")
    if results_df.empty:
        print("No results found.")
    else:
        display(results_df)

except Exception as e:
    print(f"An error occurred: {e}")

Executing BigQuery query...
Query results:


,year_month,monthly_revenue
0,2011-01,98902
1,2011-02,91152
2,2011-03,145726
3,2011-04,116927
4,2011-05,146762
5,2011-06,215214
6,2011-07,115518
7,2011-08,207570
8,2011-09,290230
9,2011-10,199070


### B2. Post‑aggregation filter (HAVING)
**Prompt:**
```
BigQuery SQL only.
Task: Find sub-categories whose total profit over the entire dataset is negative.
Table: `[YOUR_PROJECT].superstore_data.sales`
Output: `Sub_Category`, `total_profit`
Logic: SUM `Profit` GROUP BY `Sub_Category`, HAVING SUM(Profit) < 0
Sort by `total_profit` ASC (most negative first).
```
**Why HAVING?** Ask the model to include a 1-sentence explanation of why HAVING is used instead of WHERE here.

In [10]:
# Paste the BigQuery SQL generated by Vertex AI here.
# Example SQL based on the prompt:
query_string = """
SELECT
    sub_category,
    SUM(profit) AS total_profit
FROM
    `big-data-analysis-472319.superstore_data.superstore_clean`
GROUP BY
    sub_category
HAVING
    SUM(profit) < 0
ORDER BY
    total_profit ASC
"""

# Assuming 'client' object is already initialized from previous cells
# from google.cloud import bigquery
# client = bigquery.Client(project='your-gcp-project-id') # Ensure client is initialized if not already

try:
    print("Executing BigQuery query...")
    query_job = client.query(query_string)

    # Wait for the job to complete and get the results
    results_df = query_job.to_dataframe()

    print("Query results:")
    if results_df.empty:
        print("No results found.")
    else:
        display(results_df)

except Exception as e:
    print(f"An error occurred: {e}")

Executing BigQuery query...
Query results:


,sub_category,total_profit
0,Tables,-64083.3887


## Part C — Joins (dimension enrichment)
**Aim:** Use joins to enhance facts with attributes.

### C1. Join facts to a small dimension
*(If you have a customer or product dimension in your schema, use it. Otherwise, request a synthetic example.)*  
**Prompt:**
```
BigQuery SQL only.
Task: Join the sales table to a product dimension to report `Product_ID`, `Product_Name`, and total sales.
Tables: `[YOUR_PROJECT].superstore_data.sales` as s, `[YOUR_PROJECT].superstore_data.products` as p
Join key: `s.Product_ID = p.Product_ID`
Output: `Product_ID`, `Product_Name`, `total_sales`
Sort by `total_sales` DESC
```
**If you lack a dimension table:** Ask the model how to simulate one temporarily via a CTE.

In [11]:
# Paste the BigQuery SQL generated by Vertex AI here.
# Example SQL based on the prompt (assuming a simulated dimension or self-join):
query_string = """
SELECT
    product_id,
    ANY_VALUE(product_name) AS product_name, -- Assuming product_name is consistent for each product_id
    SUM(sales) AS total_sales
FROM
    `big-data-analysis-472319.superstore_data.superstore_clean`
GROUP BY
    product_id
ORDER BY
    total_sales DESC
"""

# Assuming 'client' object is already initialized from previous cells
# from google.cloud import bigquery
# client = bigquery.Client(project='your-gcp-project-id') # Ensure client is initialized if not already

try:
    print("Executing BigQuery query...")
    query_job = client.query(query_string)

    # Wait for the job to complete and get the results
    results_df = query_job.to_dataframe()

    print("Query results:")
    if results_df.empty:
        print("No results found.")
    else:
        display(results_df)

except Exception as e:
    print(f"An error occurred: {e}")

Executing BigQuery query...
Query results:


,product_id,product_name,total_sales
0,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,61600
1,TEC-PH-10004664,"Nokia Smart Phone, with Caller ID",30042
2,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,27454
3,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,22638
4,TEC-PH-10004823,"Nokia Smart Phone, Full Size",22261
...,...,...,...
10287,OFF-BI-10003253,"Ibico Index Tab, Economy",3
10288,OFF-SME-10000258,"Smead Removable Labels, 5000 Label Set",3
10289,OFF-BI-10001284,"Ibico Hole Reinforcements, Clear",3
10290,OFF-EN-10003604,"Jiffy Clasp Envelope, Set of 50",2


## Part D — Common Table Expressions (CTEs)
**Aim:** Make complex logic readable and testable in steps.

### D1. Multi‑step ranking with CTEs
**Prompt:**
```
BigQuery SQL only.
Goal: Within each `Region`, rank states by total sales and return top 3 per region.
Table: `[YOUR_PROJECT].superstore_data.sales`
CTE 1 (`state_sales`): SUM(Sales) by `Region`, `State`
CTE 2 (`ranked_state_sales`): Add `RANK() OVER (PARTITION BY Region ORDER BY total_sales DESC)` as `sales_rank`
Final SELECT: rows where `sales_rank <= 3`
Output columns: `Region`, `State`, `total_sales`, `sales_rank`
Sort: by `Region`, then `sales_rank`
```
**Ask for**: a one-paragraph explanation of each step, then **provide only the final runnable SQL**.

In [12]:
# Paste the BigQuery SQL generated by Vertex AI here.
# Example SQL based on the prompt:
query_string = """
WITH state_sales AS (
    SELECT
        region,
        state,
        SUM(sales) AS total_sales
    FROM
        `big-data-analysis-472319.superstore_data.superstore_clean`
    GROUP BY
        region,
        state
),
ranked_state_sales AS (
    SELECT
        region,
        state,
        total_sales,
        RANK() OVER (PARTITION BY region ORDER BY total_sales DESC) as sales_rank
    FROM
        state_sales
)
SELECT
    region,
    state,
    total_sales,
    sales_rank
FROM
    ranked_state_sales
WHERE
    sales_rank <= 3
ORDER BY
    region,
    sales_rank
"""

# Assuming 'client' object is already initialized from previous cells
# from google.cloud import bigquery
# client = bigquery.Client(project='your-gcp-project-id') # Ensure client is initialized if not already

try:
    print("Executing BigQuery query...")
    query_job = client.query(query_string)

    # Wait for the job to complete and get the results
    results_df = query_job.to_dataframe()

    print("Query results:")
    if results_df.empty:
        print("No results found.")
    else:
        display(results_df)

except Exception as e:
    print(f"An error occurred: {e}")

Executing BigQuery query...
Query results:


,region,state,total_sales,sales_rank
0,Africa,Gauteng,51608,1
1,Africa,Kinshasa,42533,2
2,Africa,Al Qahirah,38436,3
3,Canada,Ontario,35451,1
4,Canada,Quebec,10928,2
5,Canada,British Columbia,9546,3
6,Caribbean,Santo Domingo,78710,1
7,Caribbean,Santiago de Cuba,32456,2
8,Caribbean,Granma,16997,3
9,Central,Ile-de-France,317818,1


### D2. Time‑boxed “most improved” analysis
**Prompt:**
```
BigQuery SQL only.
Goal: Identify the top 5 sub-categories with the largest YoY revenue increase from 2023 to 2024.
Table: `[YOUR_PROJECT].superstore_data.sales`
CTE `yr_sales`: SUM(Sales) by `Sub_Category` and `year` extracted from `Order_Date`
Final: pivot or self-join to compute delta (2024 minus 2023) as `yoy_delta`
Output: `Sub_Category`, `sales_2023`, `sales_2024`, `yoy_delta`
Order by `yoy_delta` DESC
Limit 5
```
**Validation:** Ask the model for two quick failure modes (e.g., missing years) and how to handle them.

In [13]:
# Paste the BigQuery SQL generated by Vertex AI here.
# Example SQL based on the adjusted prompt (2011 to 2012):
query_string = """
WITH yr_sales AS (
    SELECT
        sub_category,
        EXTRACT(YEAR FROM PARSE_DATE('%Y-%m-%d', SUBSTR(order_date, 1, 10))) AS year,
        SUM(sales) AS total_sales
    FROM
        `big-data-analysis-472319.superstore_data.superstore_clean`
    WHERE
        EXTRACT(YEAR FROM PARSE_DATE('%Y-%m-%d', SUBSTR(order_date, 1, 10))) IN (2011, 2012)
    GROUP BY
        sub_category,
        year
)
SELECT
    s2012.sub_category,
    s2011.total_sales AS sales_2011,
    s2012.total_sales AS sales_2012,
    s2012.total_sales - s2011.total_sales AS yoy_delta
FROM
    yr_sales s2012
JOIN
    yr_sales s2011
ON
    s2012.sub_category = s2011.sub_category
WHERE
    s2012.year = 2012 AND s2011.year = 2011
ORDER BY
    yoy_delta DESC
LIMIT 5
"""

# Assuming 'client' object is already initialized from previous cells
# from google.cloud import bigquery
# client = bigquery.Client(project='your-gcp-project-id') # Ensure client is initialized if not already

try:
    print("Executing BigQuery query...")
    query_job = client.query(query_string)

    # Wait for the job to complete and get the results
    results_df = query_job.to_dataframe()

    print("Query results:")
    if results_df.empty:
        print("No results found.")
    else:
        display(results_df)

except Exception as e:
    print(f"An error occurred: {e}")

Executing BigQuery query...
Query results:


,sub_category,sales_2011,sales_2012,yoy_delta
0,Copiers,216380,327167,110787
1,Accessories,113470,172404,58934
2,Bookcases,259388,317950,58562
3,Appliances,173391,222944,49553
4,Phones,337298,364034,26736


## Part E — Window Functions (ROW_NUMBER, RANK, DENSE_RANK, LAG/LEAD, moving averages)
**Aim:** Compare rows across partitions and time; compute trends and ranks without collapsing rows.

### E1. Top product per region (ROW_NUMBER)
**Prompt:**
```
BigQuery SQL only.
Task: For each `Region`, return only the single highest-revenue `Sub_Category`.
Table: `[YOUR_PROJECT].superstore_data.sales`
CTE `subcat_sales`: SUM(Sales) by `Region`, `Sub_Category`
Add `ROW_NUMBER() OVER (PARTITION BY Region ORDER BY total_sales DESC)` as rn
Final: filter `rn = 1`
Output: `Region`, `Sub_Category`, `total_sales`
Sort by `Region`
```
**Why `ROW_NUMBER` instead of `RANK`?** Ask the model to add a 2-sentence contrast.

In the context of Task E1, ROW_NUMBER was used instead of RANK because the goal was to return only the single highest-revenue sub-category for each region. ROW_NUMBER assigns a unique, sequential integer to each row within its partition, even if there are ties in the ordering column. RANK, on the other hand, would assign the same rank to rows with identical values in the ordering column, which would return multiple rows in the case of a tie for the highest revenue in a region, if we filtered by rank = 1. By using ROW_NUMBER() ... as rn and filtering for rn = 1, we guarantee that only one row is selected per region, regardless of ties in total sales.

In [14]:
# Paste the BigQuery SQL generated by Vertex AI here.
# Example SQL based on the prompt:
query_string = """
WITH subcat_sales AS (
    SELECT
        region,
        sub_category,
        SUM(sales) AS total_sales
    FROM
        `big-data-analysis-472319.superstore_data.superstore_clean`
    GROUP BY
        region,
        sub_category
),
ranked_subcat_sales AS (
    SELECT
        region,
        sub_category,
        total_sales,
        ROW_NUMBER() OVER (PARTITION BY region ORDER BY total_sales DESC) as rn
    FROM
        subcat_sales
)
SELECT
    region,
    sub_category,
    total_sales
FROM
    ranked_subcat_sales
WHERE
    rn = 1
ORDER BY
    region
"""

# Assuming 'client' object is already initialized from previous cells
# from google.cloud import bigquery
# client = bigquery.Client(project='your-gcp-project-id') # Ensure client is initialized if not already

try:
    print("Executing BigQuery query...")
    query_job = client.query(query_string)

    # Wait for the job to complete and get the results
    results_df = query_job.to_dataframe()

    print("Query results:")
    if results_df.empty:
        print("No results found.")
    else:
        display(results_df)

except Exception as e:
    print(f"An error occurred: {e}")

Executing BigQuery query...
Query results:


,region,sub_category,total_sales
0,Africa,Phones,114830
1,Canada,Storage,10586
2,Caribbean,Copiers,49999
3,Central,Phones,370215
4,Central Asia,Phones,132715
5,EMEA,Phones,114521
6,East,Phones,100628
7,North,Phones,180419
8,North Asia,Bookcases,130070
9,Oceania,Chairs,170286


### E2. YoY growth with LAG
**Prompt:**
```
BigQuery SQL only.
Task: Compute year-over-year revenue growth for 'Phones' sub-category.
Table: `[YOUR_PROJECT].superstore_data.sales`
Steps:
- Filter to `Sub_Category = 'Phones'`
- Aggregate yearly revenue using EXTRACT(YEAR FROM Order_Date)
- Add `LAG(yearly_revenue) OVER (ORDER BY year)` as `prev_revenue`
- Compute `yoy_pct = 100.0 * (yearly_revenue - prev_revenue) / prev_revenue`
Output: `year`, `yearly_revenue`, `prev_revenue`, `yoy_pct`
Sort by `year` ASC
```
**Ask for**: a guard against divide-by-zero or NULL previous year.

In [15]:
# Paste the BigQuery SQL generated by Vertex AI here.
# Example SQL based on the prompt (adjusting for 2011-2012 data and including NULLIF):
query_string = """
WITH yearly_sales AS (
    SELECT
        EXTRACT(YEAR FROM PARSE_DATE('%Y-%m-%d', SUBSTR(order_date, 1, 10))) AS year,
        SUM(sales) AS yearly_revenue
    FROM
        `big-data-analysis-472319.superstore_data.superstore_clean`
    WHERE
        sub_category = 'Phones'
        AND EXTRACT(YEAR FROM PARSE_DATE('%Y-%m-%d', SUBSTR(order_date, 1, 10))) IN (2011, 2012) -- Filter for relevant years
    GROUP BY
        year
),
lagged_sales AS (
    SELECT
        year,
        yearly_revenue,
        LAG(yearly_revenue) OVER (ORDER BY year) as prev_revenue
    FROM
        yearly_sales
)
SELECT
    year,
    yearly_revenue,
    prev_revenue,
    CASE
        WHEN prev_revenue IS NULL OR prev_revenue = 0 THEN NULL -- Handle NULL or zero previous revenue
        ELSE 100.0 * (yearly_revenue - prev_revenue) / prev_revenue
    END AS yoy_pct
FROM
    lagged_sales
ORDER BY
    year ASC
"""

# Assuming 'client' object is already initialized from previous cells
# from google.cloud import bigquery
# client = bigquery.Client(project='your-gcp-project-id') # Ensure client is initialized if not already

try:
    print("Executing BigQuery query...")
    query_job = client.query(query_string)

    # Wait for the job to complete and get the results
    results_df = query_job.to_dataframe()

    print("Query results:")
    if results_df.empty:
        print("No results found.")
    else:
        display(results_df)

except Exception as e:
    print(f"An error occurred: {e}")

Executing BigQuery query...
Query results:


,year,yearly_revenue,prev_revenue,yoy_pct
0,2011,337298,<NA>,NaN
1,2012,364034,337298,7.926522


### E3. 3‑month moving average (MA)
**Prompt:**
```
BigQuery SQL only.
Task: For the 'Corporate' segment, compute a 3-month moving average of monthly revenue.
Table: `[YOUR_PROJECT].superstore_data.sales`
Steps:
- Derive `month` via DATE_TRUNC(Order_Date, MONTH)
- SUM(Sales) per `month`
- Add `AVG(monthly_revenue) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` as `ma_3`
Output: `month`, `monthly_revenue`, `ma_3`
Sort by `month` ASC
```
**Tip:** Ask the model to include a 1‑line cost control note (e.g., restrict date range while iterating).

In [16]:
# Paste the BigQuery SQL generated by Vertex AI here.
# Example SQL based on the prompt (using a subquery):
query_string = """
SELECT
    month,
    monthly_revenue_float,
    AVG(monthly_revenue_float) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS ma_3
FROM (
    SELECT
        DATE_TRUNC(PARSE_DATE('%Y-%m-%d', SUBSTR(order_date, 1, 10)), MONTH) AS month,
        CAST(SUM(sales) AS FLOAT64) AS monthly_revenue_float -- Calculate sum and cast in the subquery
    FROM
        `big-data-analysis-472319.superstore_data.superstore_clean`
    WHERE
        segment = 'Corporate'
        -- Add a WHERE clause here to restrict date range for cost control, e.g.:
        AND PARSE_DATE('%Y-%m-%d', SUBSTR(order_date, 1, 10)) BETWEEN PARSE_DATE('%Y-%m-%d', '2011-01-01') AND PARSE_DATE('%Y-%m-%d', '2012-12-31')
    GROUP BY
        month
)
ORDER BY
    month ASC
"""

# Assuming 'client' object is already initialized from previous cells
# from google.cloud import bigquery
# client = bigquery.Client(project='your-gcp-project-id') # Ensure client is initialized if not already

try:
    print("Executing BigQuery query...")
    query_job = client.query(query_string)

    # Wait for the job to complete and get the results
    results_df = query_job.to_dataframe()

    print("Query results:")
    if results_df.empty:
        print("No results found.")
    else:
        display(results_df)

except Exception as e:
    print(f"An error occurred: {e}")

Executing BigQuery query...
Query results:


,month,monthly_revenue_float,ma_3
0,2011-01-01,24281.0,24281.000000
1,2011-02-01,42997.0,33639.000000
2,2011-03-01,31919.0,33065.666667
3,2011-04-01,41021.0,38645.666667
4,2011-05-01,54163.0,42367.666667
5,2011-06-01,68107.0,54430.333333
6,2011-07-01,30191.0,50820.333333
7,2011-08-01,69391.0,55896.333333
8,2011-09-01,75647.0,58409.666667
9,2011-10-01,62713.0,69250.333333


## Part F — Debugging & Optimization Prompts
**Aim:** Use the model as a rubber duck for error handling and performance.

F1. Reduce cost / improve speed
**Prompt:**
```
Act as a BigQuery cost optimizer.
Given this query (below), list 3 ways to reduce scanned bytes and improve performance without changing the business logic.
[PASTE YOUR SQL HERE]
Prioritize: partition filters, column pruning, pre-aggregations, and temporary results via CTEs.
```

Leverage Partitioning (if applicable): If the superstore_data.superstore_clean table is partitioned by order_date (or a date-based column), the WHERE EXTRACT(YEAR FROM PARSE_DATE('%Y-%m-%d', SUBSTR(order_date, 1, 10))) IN (2011, 2012) clause will allow BigQuery to only scan the relevant partitions for those years. This drastically reduces the amount of data read. Action: Ensure the table is partitioned by a date column and verify the query is using a filter on that partition column or a derivative like EXTRACT(YEAR) that BigQuery can use for partition pruning.
Column Pruning: The query only explicitly uses order_date, sub_category, and sales from the base table. Ensure that the SELECT statement within the yearly_sales CTE only selects these necessary columns. While BigQuery is generally good at column pruning automatically, explicitly selecting only needed columns is a good practice and can sometimes help, especially with complex schemas or nested data. Action: Modify the SELECT in the yearly_sales CTE to only include order_date, sub_category, and sales.
Optimize Date Extraction and Filtering: The use of PARSE_DATE('%Y-%m-%d', SUBSTR(order_date, 1, 10)) within both EXTRACT(YEAR FROM ...) and the WHERE clause adds computation. If order_date could be stored as a native DATE or TIMESTAMP type in the source table or view, this parsing would be unnecessary, simplifying the query and potentially improving performance. Action: If possible, modify the data ingestion or view creation process to store order_date as a native BigQuery DATE or TIMESTAMP type. If not, ensure the SUBSTR and PARSE_DATE logic is as efficient as possible, although the primary optimization here is data type storage.

## Part G — Validation & Counter‑examples (DIVE: Validate)
**Aim:** Avoid “first‑answer fallacy” by testing alternatives.

### G1. Ask for counter‑queries
**Prompt:**
```
I concluded that 'Tables' is a high‑sales but negative‑profit sub-category due to high discounts.
Create two alternative BigQuery SQL queries that could falsify or nuance this finding:
- One that slices by region and time
- One that controls for order priority or ship mode
Return BigQuery SQL only, then a one-paragraph note on how to compare outcomes.
```

In [17]:
# Paste the BigQuery SQL generated by Vertex AI here.
# Example SQL based on the prompt:
query_string = r"""
-- Counter-query 1: Slice by Region and Year to see if the trend is consistent
SELECT
    region,
    EXTRACT(YEAR FROM PARSE_DATE('%Y-%m-%d', SUBSTR(order_date, 1, 10))) AS year,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,
    AVG(discount) AS average_discount
FROM
    `big-data-analysis-472319.superstore_data.superstore_clean`
WHERE
    sub_category = 'Tables'
GROUP BY
    region,
    year
ORDER BY
    region,
    year;
"""

# Assuming 'client' object is already initialized from previous cells
# from google.cloud import bigquery
# client = bigquery.Client(project='your-gcp-project-id') # Ensure client is initialized if not already

try:
    print("Executing BigQuery query...")
    query_job = client.query(query_string)

    # Wait for the job to complete and get the results
    results_df = query_job.to_dataframe()

    print("Query results:")
    if results_df.empty:
        print("No results found.")
    else:
        display(results_df)

except Exception as e:
    print(f"An error occurred: {e}")

Executing BigQuery query...
Query results:


,region,year,total_sales,total_profit,average_discount
0,Africa,2011,7683,936.7560,0.200000
1,Africa,2012,1446,-1455.3240,0.350000
2,Africa,2013,7972,1980.6600,0.000000
3,Africa,2014,17430,2548.5930,0.133333
4,Canada,2013,850,300.1800,0.000000
5,Caribbean,2011,443,-857.3700,0.700000
6,Caribbean,2012,6449,1300.4960,0.233333
7,Caribbean,2013,4086,-826.4240,0.350000
8,Caribbean,2014,12682,446.6860,0.250000
9,Central,2011,22669,-2507.6695,0.337931


In [18]:
# Paste the BigQuery SQL generated by Vertex AI here.
# Example SQL based on the prompt:
query_string = r"""
-- Counter-query 2: Analyze by Ship Mode to see its impact on Sales, Profit, and Discount
SELECT
    ship_mode,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,
    AVG(discount) AS average_discount,
    COUNT(*) as order_count
FROM
    `big-data-analysis-472319.superstore_data.superstore_clean`
WHERE
    sub_category = 'Tables'
GROUP BY
    ship_mode
ORDER BY
    total_sales DESC;
"""

# Assuming 'client' object is already initialized from previous cells
# from google.cloud import bigquery
# client = bigquery.Client(project='your-gcp-project-id') # Ensure client is initialized if not already

try:
    print("Executing BigQuery query...")
    query_job = client.query(query_string)

    # Wait for the job to complete and get the results
    results_df = query_job.to_dataframe()

    print("Query results:")
    if results_df.empty:
        print("No results found.")
    else:
        display(results_df)

except Exception as e:
    print(f"An error occurred: {e}")

Executing BigQuery query...
Query results:


,ship_mode,total_sales,total_profit,average_discount,order_count
0,Standard Class,452630,-36080.9019,0.293888,517
1,Second Class,158444,-20877.6052,0.306608,171
2,First Class,124220,-8093.2286,0.270504,139
3,Same Day,21740,968.3470,0.245588,34


To compare the outcomes and falsify or nuance your initial finding:

*   **Query 1 (Region and Year):**  The results show that while 'Tables' generally have negative profit across many regions and years, there are instances where they are profitable (e.g., Africa in 2011, 2013, 2014; Canada in 2013; Caribbean in 2012; Central Asia in 2012, 2013, 2014; EMEA in 2011, 2012, 2013; North in all years; Oceania in 2011, 2012, 2013; West in 2011, 2012, 2014). This significantly nuances the finding that discounts always lead to negative profit for 'Tables'. It suggests that profitability is highly dependent on the region and year, and factors beyond just the discount percentage in isolation (like regional pricing, demand, or operational costs specific to those regions/years) are clearly at play.
*   **Query 2 (Ship Mode):** The results show that 'Tables' are unprofitable for 'Standard Class', 'Second Class', and 'First Class' ship modes, which account for the vast majority of sales and orders. However, the 'Same Day' ship mode shows a positive profit, despite having an average discount similar to or lower than the other modes. This suggests that operational costs or pricing strategies associated with different ship modes, or perhaps the customer segment using 'Same Day' shipping, significantly impact the profitability of 'Tables', potentially more so than the discount level alone across all ship modes.

## Part H — Synthesis (DIVE: Extend)
**Aim:** Turn analysis into business‑ready insights.

### H1. Executive‑style summary
**Prompt:**
```
Act as a business strategist.
Based on the following metrics/figures (briefly summarize your results here), write a 4-sentence executive summary:
- 1 sentence: what changed and by how much
- 1 sentence: why it likely changed (drivers)
- 1 sentence: recommended action (who/what/when)
- 1 sentence: metric to monitor next
```

Initial analysis showed that despite high sales, the 'Tables' sub-category is overall unprofitable, with a significant total profit loss across the dataset. While high discounts contribute, further analysis revealed that profitability varies significantly by region/year and is positive for the 'Same Day' ship mode, indicating that operational factors and regional specifics are also key drivers. To improve profitability, we should investigate pricing and operational costs for 'Tables' in underperforming regions and for standard shipping methods within the next quarter. We should continue to monitor 'Tables' profitability segmented by region and ship mode, as well as the average discount applied, to track the impact of any changes.

### H2. Convert final SQL into an automated job (optional)
**Prompt (use only after your SQL is final):**
```
Convert my final BigQuery SQL into a Python script that can run as a scheduled job from Colab or Cloud Functions.
Requirements:
- Use python‑bigquery client
- Parameterize date range
- Write results to a destination table `[YOUR_PROJECT].analytics.outputs_kpi`
- Add basic error handling & logging
Return one complete runnable script.
```

---
## Submission checklist
- [ ] Kept prompts precise and reproducible  
- [ ] Captured at least **one** CTE query and **one** window function query  
- [ ] Documented **two** validation attempts (counter‑queries or alternate slice)  
- [ ] Wrote a 4‑sentence executive summary based on results  
- [ ] (Optional) Converted final query into a scheduled job
---